In [ ]:
# 1. Загрузка данных, метрики, роли, кластеры и приоритеты
from pathlib import Path
import pandas as pd
import networkx as nx
from IPython.display import display
import starter
from starter import (
    load, build_graph, basic_features, assign_roles,
    assign_clusters_and_priority, write_outputs,
)

PROJECT_ROOT = Path(starter.__file__).resolve().parent
OUT_DIR = PROJECT_ROOT / "out"
edges, nodes, tx = load(PROJECT_ROOT / "data")
G = build_graph(edges)
df = assign_roles(basic_features(G, nodes), G)
df, clusters = assign_clusters_and_priority(df, G)
write_outputs(df, OUT_DIR, clusters)
print(f"Узлов: {len(df)}, рёбер: {G.number_of_edges()}, кластеров: {len(clusters)}")


In [ ]:
# 2. Топ-25 узлов; gid читаем строкой для сохранения всех цифр
top_nodes = pd.read_csv(OUT_DIR / "top_nodes.csv", dtype={"gid": str})
display(top_nodes.head(25))


In [ ]:
# 3. Интерактивный граф; установка в окружение текущего ядра при необходимости
import importlib.util
import subprocess
import sys

if importlib.util.find_spec("pyvis") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "pyvis"])

from pyvis.network import Network

ROLE_COLORS = {
    "consolidator": "red",
    "distributor": "orange",
    "transit": "yellow",
    "terminal": "blue",
    "coordinator": "purple",
    "peripheral": "gray",
}
node_rows = {str(row["gid"]): row for row in df.to_dict("records")}


def populate_network(net, graph, extra_nodes=()):
    # Строковые ID защищают 18-значные gid от округления JavaScript.
    for gid in dict.fromkeys([*graph.nodes, *extra_nodes]):
        row = node_rows[str(gid)]
        net.add_node(
            str(gid), label=str(gid), title=row["evidence"],
            color=ROLE_COLORS[row["role"]],
            size=18 if row["is_seed"] else 10,
        )
    for src, dst, attrs in graph.edges(data=True):
        net.add_edge(
            str(src), str(dst),
            title=f"sum_kzt={attrs['sum_kzt']:,.2f}; n_tx={attrs.get('n_tx', 0)}",
        )
    net.barnes_hut()
    net.set_options('{"physics": {"stabilization": {"iterations": 150}}}')
    return net


net = Network(directed=True, notebook=True, cdn_resources="in_line")
# Включаем также изоляты из df, которых нет в G.
populate_network(net, G, extra_nodes=df["gid"])
display(net.show("graph.html"))


In [ ]:
# 4. Поиск узла и направленный ego-граф радиуса 2
# nx.ego_graph по умолчанию следует исходящим рёбрам.
def search_node(gid):
    key = str(gid).strip()
    matches = df.loc[df["gid"].astype(str).eq(key)]
    if matches.empty:
        print(f"Узел {key} не найден. Передавайте gid строкой или целым числом.")
        return

    display(matches)
    graph_gid = matches["gid"].iloc[0]
    if graph_gid in G:
        ego = nx.ego_graph(G, graph_gid, radius=2)
    else:
        # Изолированные seed тоже доступны для поиска.
        ego = nx.DiGraph()
        ego.add_node(graph_gid)

    ego_net = Network(directed=True, notebook=True, cdn_resources="in_line")
    populate_network(ego_net, ego)
    ego_net.get_node(key)["size"] = 25
    display(ego_net.show(f"ego_{int(graph_gid)}.html"))


In [ ]:
# 5. Пример: первый seed-узел
seed_gids = df.loc[df["is_seed"], "gid"]
if not seed_gids.empty:
    search_node(str(seed_gids.iloc[0]))
else:
    print("В данных нет seed-узлов.")
